In [2]:

import re
from collections import defaultdict

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT = {'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,'risorse':5,'stabilita':5,
           'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
           'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,
           'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,'risorse_russia':5,
           'influenza_militare_russia':5,'veto_onu_russia':2,'stabilita_economica_russia':5,
           'stabilita_russia':5,'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,
           'stabilita_rotte_cina':5,'stabilita_cina':5,'risorse_europa':5,'influenza_diplomatica_europa':5,
           'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5}
ALL_T = list(DEFAULT.keys())

def eval_ternary(expr, v):
    expr = expr.strip().rstrip(',')
    def replace_ternary(e):
        pattern = re.compile(r'([^?:]+)\?([^?:]+):([^?:]+)')
        while '?' in e:
            e = pattern.sub(lambda m: f"({m.group(2)} if {m.group(1)} else {m.group(3)})", e, count=1)
        return e
    try:
        return eval(replace_ternary(expr), {'v': v})
    except:
        return None

card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)
track_pos = defaultdict(int)
track_neg = defaultdict(int)

for m in card_re.finditer(src):
    cid, cname, faz, ctype, eff_str = m.groups()
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            r = eval_ternary(fn.group(1), DEFAULT[t])
            if r is not None:
                d = float(r) - DEFAULT[t]
                if d > 0: track_pos[t] += 1
                elif d < 0: track_neg[t] += 1

print("=== VERIFICA FINALE BIDIREZIONALITÀ ===")
final_results = []
all_ok = True
for t in ALL_T:
    pos = track_pos[t]; neg = track_neg[t]
    ok = pos > 0 and neg > 0
    sym = "✅" if ok else "❌"
    if not ok: all_ok = False
    final_results.append((t, pos, neg, ok))
    print(f"  {sym} {t}: +{pos} / -{neg}")

print(f"\nTutti bidirezionali: {all_ok}")
print(f"Final results salvati in memoria")

# Salva per uso successivo
import json
with open('/tmp/final_bidir.json', 'w') as f:
    json.dump(final_results, f)


=== VERIFICA FINALE BIDIREZIONALITÀ ===
  ✅ nucleare: +28 / -48
  ✅ sanzioni: +80 / -97
  ✅ opinione: +130 / -22
  ✅ defcon: +54 / -102
  ✅ risorse: +90 / -60
  ✅ stabilita: +70 / -61
  ✅ risorse_iran: +8 / -22
  ✅ forze_militari_iran: +89 / -28
  ✅ tecnologia_nucleare_iran: +11 / -25
  ✅ stabilita_iran: +38 / -49
  ✅ risorse_coalizione: +7 / -10
  ✅ influenza_diplomatica_coalizione: +36 / -46
  ✅ tecnologia_avanzata_coalizione: +17 / -16
  ✅ supporto_pubblico_coalizione: +35 / -12
  ✅ stabilita_coalizione: +7 / -50
  ✅ risorse_russia: +16 / -4
  ✅ influenza_militare_russia: +33 / -24
  ✅ veto_onu_russia: +23 / -9
  ✅ stabilita_economica_russia: +15 / -5
  ✅ stabilita_russia: +9 / -4
  ✅ risorse_cina: +8 / -32
  ✅ influenza_commerciale_cina: +55 / -14
  ✅ cyber_warfare_cina: +8 / -9
  ✅ stabilita_rotte_cina: +29 / -5
  ✅ stabilita_cina: +9 / -3
  ✅ risorse_europa: +9 / -21
  ✅ influenza_diplomatica_europa: +50 / -15
  ✅ aiuti_umanitari_europa: +5 / -3
  ✅ coesione_ue_europa: +36 / -63
